# MLP multi-trait: vector fenotipico + mascara real/imputado

Extiende `03_prediction/mlp_temporal_v2.ipynb` en dos ejes, siguiendo el mismo esquema temporal (ventana creciente: entrenar con anios < t, validar/elegir epocas en t-1, probar en t):

1. **Salida multi-trait**: en vez de predecir solo YLD_BE, la cabeza de la red predice las 8 columnas `BLUE_<trait>` a la vez (ERM, MST, PHT, RTLP, STLP, TWT, YLD_BE, EHT), compartiendo el tronco de la red. La cobertura de cada trait es muy distinta (MST/YLD_BE ~96-97% de las lineas, PHT/EHT solo ~22-29%), asi que la funcion de perdida enmascara por trait: una linea sin PHT medido no penaliza esa salida.
2. **Mascara real/imputado**: `real_mask.npy` (construida en `build_dataset.py` a partir del genotipo SIN imputar) indica que SNPs fueron observados de verdad (~2-3% del total por linea) vs reconstruidos por imputacion (~97-98%). Con `use_mask=True` se concatena esa mascara al genotipo como un segundo bloque de features (duplica el ancho de entrada), para que la red pueda aprender a confiar menos en los valores imputados.

Variantes comparadas: `no_mask` vs `with_mask` (mismo MLP, unica diferencia es si la mascara entra como feature).

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


def find_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "DIGITALTOOL").is_dir():
            return d
    raise FileNotFoundError("Run the notebook from inside the Hackolombia repo")


BASE_DIR = find_root(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "DIGITALTOOL" / "data" / "processed" / "multi_trait_prediction"
sys.path.insert(0, str(BASE_DIR / "DIGITALTOOL" / "src" / "03_prediction"))
from ridge_temporal import pearson, metrics  # same metrics as the ridge baselines

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

TRAITS = ["ERM", "MST", "PHT", "RTLP", "STLP", "TWT", "YLD_BE", "EHT"]
N_TRAITS = len(TRAITS)

In [ ]:
QUICK_RUN = True   # True: test year 2008 only, 1 seed, few epochs (smoke test)

BASE = dict(
    max_snp_na=0.20,
    test_years=[2008] if QUICK_RUN else [2006, 2007, 2008],
    arch="mlp",              # "mlp" | "resnet"
    width=256,
    n_blocks=2,              # residual blocks (resnet only)
    dropout=0.3,
    input_dropout=0.1,
    use_mask=True,           # concat the real/imputed SNP mask to the genotype
    use_parents=False,
    parent_dim=16,
    parent_index_dropout=0.2,
    lr=1e-3,
    weight_decay=1e-2,
    batch_size=512,
    max_epochs=5 if QUICK_RUN else 40,
    patience=5,
    n_seeds=1 if QUICK_RUN else 3,
    use_weights=True,        # weight lines by number of observations (per trait)
    weight_cap=10,
)
VARIANTS = {
    "no_mask": {"use_mask": False},
    "with_mask": {"use_mask": True},
}

## 1. Datos

In [ ]:
X_raw = np.load(DATA_DIR / "X.npy", mmap_mode="r")
mask_raw = np.load(DATA_DIR / "real_mask.npy", mmap_mode="r")
lines = pd.read_parquet(DATA_DIR / "lines.parquet")

keep = np.where(np.isnan(X_raw).mean(axis=0) < BASE["max_snp_na"])[0]
X = np.asarray(X_raw[:, keep], dtype=np.float32)
M = np.asarray(mask_raw[:, keep], dtype=np.float32)
print(f"{X.shape[0]} lines, {X.shape[1]}/{X_raw.shape[1]} SNPs kept | real-marker fraction (kept SNPs): {M.mean():.4f}")

# Mean imputation + standardization using pre-2008 lines only (2008 is never used for fitting)
years = lines["YEAR"].to_numpy()
pre = years < 2008
mu = np.nanmean(X[pre], axis=0)
for j in range(X.shape[1]):
    col = X[:, j]
    col[np.isnan(col)] = mu[j]
X -= mu
sd = X[pre].std(axis=0)
sd[sd < 1e-6] = 1.0
X /= sd
print("NaN left in X:", int(np.isnan(X).sum()))

X_t = torch.from_numpy(X)
M_t = torch.from_numpy(M)  # 0/1, no imputation needed


def features(idx_t, use_mask):
    """idx_t: LongTensor of row indices (CPU). Returns the model input for those rows."""
    Xb = X_t[idx_t]
    return torch.cat([Xb, M_t[idx_t]], dim=1) if use_mask else Xb

In [ ]:
# Targets: one BLUE per trait, each year-centered and scaled by its own pre-2008 sd.
# obs: which (line, trait) pairs actually have a measurement (coverage varies a lot by trait).
Y = np.full((len(lines), N_TRAITS), np.nan, dtype=np.float32)
obs = np.zeros((len(lines), N_TRAITS), dtype=bool)
y_sd = {}
for j, t in enumerate(TRAITS):
    col = lines[f"BLUE_{t}"]
    centered = col - lines.groupby("YEAR")[f"BLUE_{t}"].transform("mean")
    sd_t = float(centered[pre].std())
    y_sd[t] = sd_t
    obs[:, j] = col.notna().to_numpy()
    Y[obs[:, j], j] = (centered / sd_t).to_numpy(dtype=np.float32)[obs[:, j]]
Y = np.nan_to_num(Y, nan=0.0)  # masked out in the loss anyway

pops = lines["POP"].to_numpy()

# Per-trait observation weight (more observations -> more reliable BLUE), same idea as stage 03/single-trait.
W = np.ones((len(lines), N_TRAITS), dtype=np.float32)
for j, t in enumerate(TRAITS):
    w = np.clip(lines[f"N_OBS_{t}"].fillna(1).to_numpy(dtype=np.float32), 1, BASE["weight_cap"])
    W[:, j] = w / w[obs[:, j]].mean() if obs[:, j].any() else 1.0

Y_t = torch.from_numpy(Y)
obs_t = torch.from_numpy(obs)
W_t = torch.from_numpy(W)

# Parent ids per line (0 is reserved for "unknown / unseen in training")
par_tab = pd.read_csv(DATA_DIR / "parents.csv").set_index("POP")
pid = {p: i + 1 for i, p in enumerate(pd.unique(par_tab[["PARENT1", "PARENT2"]].to_numpy().ravel()))}
N_PARENTS = len(pid)
parent_idx = np.stack([lines["POP"].map(par_tab["PARENT1"]).map(pid).fillna(0),
                       lines["POP"].map(par_tab["PARENT2"]).map(pid).fillna(0)], axis=1).astype(np.int64)

## 2. Modelo y entrenamiento

Mismo esquema que `mlp_temporal_v2.ipynb`: proyeccion -> (opcional) bloques residuales -> (opcional) embedding de padres -> cabeza. Unica diferencia: la cabeza tiene `N_TRAITS` salidas en vez de 1, y la perdida enmascara por trait (`obs`) ademas de ponderar por confiabilidad (`W`).

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, d, p):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, d))

    def forward(self, x):
        return x + self.net(x)


class GenoNet(nn.Module):
    def __init__(self, cfg, n_in):
        super().__init__()
        wd, dr = cfg["width"], cfg["dropout"]
        self.drop_in = nn.Dropout(cfg["input_dropout"])
        self.proj = nn.Sequential(nn.Linear(n_in, wd), nn.GELU(), nn.Dropout(dr))
        if cfg["arch"] == "resnet":
            self.body = nn.Sequential(*[ResBlock(wd, dr) for _ in range(cfg["n_blocks"])])
        else:
            self.body = nn.Identity()
        self.use_parents = cfg["use_parents"]
        head_in = wd
        if self.use_parents:
            self.emb = nn.Embedding(N_PARENTS + 1, cfg["parent_dim"], padding_idx=0)
            head_in += cfg["parent_dim"]
        self.head = nn.Sequential(nn.LayerNorm(head_in), nn.Linear(head_in, 64), nn.GELU(), nn.Linear(64, N_TRAITS))
        self.seen = np.array([], dtype=np.int64)

    def forward(self, x, par):
        h = self.body(self.proj(self.drop_in(x)))
        if self.use_parents:
            h = torch.cat([h, self.emb(par).sum(dim=1)], dim=1)
        return self.head(h)


def mask_parents(idx, seen):
    p = parent_idx[idx]
    return np.where(np.isin(p, seen), p, 0)


def multi_metrics(pred, y, ob, pop):
    """Per-trait metrics() (imported from the ridge baseline) + a mean r_all used for early stopping."""
    out, r_alls = {}, []
    for j, t in enumerate(TRAITS):
        m = ob[:, j]
        if m.sum() < 20:
            continue
        out[t] = metrics(pred[m, j], y[m, j].astype(np.float64), pop[m])
        if not np.isnan(out[t]["r_all"]):
            r_alls.append(out[t]["r_all"])
    out["_mean_r_all"] = float(np.mean(r_alls)) if r_alls else np.nan
    return out


@torch.no_grad()
def predict(model, idx, use_mask):
    model.eval()
    idx_t = torch.from_numpy(idx)
    par = torch.from_numpy(mask_parents(idx, model.seen))
    Xq = features(idx_t, use_mask)
    out = [model(Xq[i:i + 4096].to(DEVICE), par[i:i + 4096].to(DEVICE)).cpu() for i in range(0, len(idx), 4096)]
    return torch.cat(out).numpy()


def train_model(cfg, tr_idx, val_idx=None, epochs=None, seed=0):
    """Train on lines tr_idx, epoch by epoch over shuffled mini-batches.
    With val_idx: early-stop on mean validation Pearson r across traits (returns the best epoch).
    With epochs: train exactly that many epochs."""
    torch.manual_seed(seed)
    n_in = X.shape[1] * (2 if cfg["use_mask"] else 1)
    model = GenoNet(cfg, n_in).to(DEVICE)
    model.seen = np.unique(parent_idx[tr_idx])
    model.seen = model.seen[model.seen > 0]

    tr_idx_t = torch.from_numpy(tr_idx)
    Xtr = features(tr_idx_t, cfg["use_mask"]).to(DEVICE)
    ytr = Y_t[tr_idx_t].to(DEVICE)
    obtr = obs_t[tr_idx_t].to(DEVICE)
    wtr = W_t[tr_idx_t].to(DEVICE)
    ptr = torch.from_numpy(parent_idx[tr_idx]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    bs, n = cfg["batch_size"], len(tr_idx)
    max_ep = epochs if epochs is not None else cfg["max_epochs"]
    best_r, best_ep, hist = -np.inf, 1, []

    for ep in range(1, max_ep + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        loss_sum = 0.0
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            pb = ptr[b]
            if model.use_parents:
                hide = torch.rand(len(b), 1, device=DEVICE) < cfg["parent_index_dropout"]
                pb = pb.masked_fill(hide, 0)
            pred = model(Xtr[b], pb)
            wt = obtr[b].float() * wtr[b]
            loss = (wt * (pred - ytr[b]) ** 2).sum() / wt.sum().clamp_min(1.0)
            opt.zero_grad()
            loss.backward()
            opt.step()
            loss_sum += loss.item() * len(b)
        row = {"epoch": ep, "train_loss": loss_sum / n}
        if val_idx is not None:
            pv = predict(model, val_idx, cfg["use_mask"])
            m = multi_metrics(pv, Y[val_idx], obs[val_idx], pops[val_idx])
            row.update(val_r_all=m["_mean_r_all"])
            if m["_mean_r_all"] > best_r:
                best_r, best_ep = m["_mean_r_all"], ep
        hist.append(row)
        if val_idx is not None and ep - best_ep >= cfg["patience"]:
            break
    return model, best_ep, pd.DataFrame(hist)

## 3. Ventana creciente por variante

Para cada anio de prueba t: (1) por semilla, entrenar con anios < t-1 y validar en t-1 (r promedio entre traits decide el numero de epocas); (2) reentrenar con anios < t y promediar las semillas. El anio t no decide nada.

In [ ]:
def run_variant(name, cfg):
    rows, curves, preds = [], [], {}
    for t in cfg["test_years"]:
        t0 = time.time()
        tr_prev = np.where(years < t - 1)[0]
        val = np.where(years == t - 1)[0]
        tr_all = np.where(years < t)[0]
        te = np.where(years == t)[0]

        best_eps = []
        for s in range(cfg["n_seeds"]):
            _, ep, hist = train_model(cfg, tr_prev, val_idx=val, seed=s)
            best_eps.append(ep)
            curves.append(hist.assign(variant=name, test_year=t, seed=s))
        n_ep = int(np.median(best_eps))

        seed_preds = []
        for s in range(cfg["n_seeds"]):
            model, _, _ = train_model(cfg, tr_all, epochs=n_ep, seed=100 + s)
            seed_preds.append(predict(model, te, cfg["use_mask"]))
        pred = np.mean(seed_preds, axis=0)

        m = multi_metrics(pred, Y[te], obs[te], pops[te])
        for trait, mt in m.items():
            if trait == "_mean_r_all":
                continue
            row = dict(mt)
            row.update(variant=name, trait=trait, test_year=t, n_epochs=n_ep, n_train=len(tr_all), n_test=len(te))
            rows.append(row)
        preds[t] = pd.DataFrame({"LINE_ID": lines["LINE_ID"].to_numpy()[te], "POP": pops[te],
                                 **{f"pred_{tr}": pred[:, j] for j, tr in enumerate(TRAITS)},
                                 **{f"y_{tr}": Y[te, j] for j, tr in enumerate(TRAITS)}})
        print(f"[{name}] test {t}: epochs={n_ep} (per seed {best_eps}) | mean r_all={m['_mean_r_all']:.3f} | "
              f"{time.time() - t0:.0f}s")
    return pd.DataFrame(rows), pd.concat(curves, ignore_index=True), preds


results, curves, preds = [], [], {}
for name, over in VARIANTS.items():
    r, c, p = run_variant(name, {**BASE, **over})
    results.append(r); curves.append(c); preds[name] = p
results = pd.concat(results, ignore_index=True)
curves = pd.concat(curves, ignore_index=True)

## 4. Resultados: efecto de la mascara, por trait y por anio

In [ ]:
# Rescale gains to each trait's own units and compare no_mask vs with_mask.
res = results.copy()
for trait in TRAITS:
    m = res["trait"] == trait
    for c in ("gain_all", "oracle_gain_all", "gain_within", "oracle_gain_within"):
        res.loc[m, c] = res.loc[m, c] * y_sd[trait]

pivot = res.pivot_table(index=["trait", "test_year"], columns="variant", values=["r_all", "r_within"])
print(pivot.round(3).to_string())

res.to_csv(DATA_DIR / "mlp_multitrait_results.csv", index=False)
print(f"\nSaved {DATA_DIR / 'mlp_multitrait_results.csv'}")

In [ ]:
fig, axes = plt.subplots(1, len(TRAITS), figsize=(3.2 * len(TRAITS), 3.2), sharey=True)
for ax, trait in zip(axes, TRAITS):
    d = res[res["trait"] == trait]
    for name, dd in d.groupby("variant"):
        ax.plot(dd["test_year"], dd["r_all"], "o-", label=name)
    ax.set_title(trait); ax.axhline(0, color="gray", lw=0.5)
    ax.set_xticks(sorted(d["test_year"].unique()))
axes[0].set_ylabel("r_all"); axes[0].legend()
plt.suptitle("r_all por trait: no_mask vs with_mask", y=1.05)
plt.tight_layout(); plt.show()